# 01: The raw material

Free data is the only reason this project exists. StatsBomb publishes event data for hundreds of matches, and EA's FC 24 database has weekly wages for every player. Same season on both sides: 2023/24. No scraping, no API keys, everything reproducible with one download script.

In [1]:
from effc.data.statsbomb import list_matches
from effc.data.download import MODEL_POOL, TARGET

pool = list_matches(MODEL_POOL)
target = list_matches([TARGET])
print(f"model pool: {len(pool)} matches")
print(f"target (ratings): {len(target)} matches")
print(target[['match_date','home_team','away_team']].head(3).to_string(index=False))

model pool: 196 matches
target (ratings): 34 matches
match_date                home_team        away_team
2023-08-19         Bayer Leverkusen       RB Leipzig
2023-08-26 Borussia Mönchengladbach Bayer Leverkusen
2023-09-02         Bayer Leverkusen     Darmstadt 98


The model pool trains the goal-probability models. Goal probability given a game state does not care much which competition it is, so pooling Bundesliga seasons with World Cups buys a lot more training data. Ratings are Bundesliga 2023/24 only, because that is the season the wages match.

In [2]:
from effc.data.statsbomb import load_events

events = load_events(int(target['match_id'].iloc[0]))
print(f"{len(events)} events in one match")
import pandas as pd
types = pd.Series([e['type']['name'] for e in events]).value_counts()
print(types.head(8).to_string())

3739 events in one match
Pass             1081
Ball Receipt*    1019
Carry             892
Pressure          297
Ball Recovery      87
Duel               53
Goal Keeper        33
Block              32


Now the money side. FC 24 wages, matched to StatsBomb players by name. This is the fiddliest part of the whole project: footballers have inconsistent name spellings across datasets, so it is exact match first, then last-name plus initial within the same club.

In [3]:
from effc.data.wages import load_wages, match_players
from effc.vaep.features import featurize_match
from effc.data.statsbomb import load_events
import pandas as pd

wages = load_wages()
print(f"{len(wages)} players with wages")
# player pool from one target match, just to demo the join
pm_demo = []
for mid in target['match_id'][:5]:
    df = featurize_match(load_events(int(mid)), match_id=int(mid))
    pm_demo.append(df[['player','team']].drop_duplicates())
sb = pd.concat(pm_demo).drop_duplicates().reset_index(drop=True)
m = match_players(sb, wages)
print(f"matched: {m['matched'].mean():.1%}")
print(m[m['matched']][['player','team','wage_eur','pos_group']].head(8).to_string(index=False))

18416 players with wages


matched: 83.2%
              player       team  wage_eur pos_group
Daniel Olmo Carvajal RB Leipzig   71000.0       MID
   Benjamin Henrichs RB Leipzig   53000.0       DEF
     Mohamed Simakan RB Leipzig   46000.0       DEF
         Willi Orban RB Leipzig   67000.0       DEF
          David Raum RB Leipzig   51000.0       DEF
         Timo Werner RB Leipzig   63000.0       FWD
         Xavi Simons RB Leipzig   71000.0       MID
      Janis Blaswich RB Leipzig   37000.0        GK


88% match rate on the full run. The misses are mostly youth players with few minutes, which the minutes filter drops anyway. Good enough to build on.